# Explainable Multi-System Risk Modeling of Smoking Exposure using NHANES

## Notebook Aim
This Step-1 notebook develops a Q1-oriented, explainable, and decision-aware modeling pipeline using the Step-0 NHANES project dataset.

## Main Objectives
- Load the Step-0 NHANES project dataset automatically from Google Drive
- Validate the integrated dataset structure
- Rebuild derived targets automatically if they are missing from the CSV
- Build a smoking-exposure classification task
- Build organ-specific prediction tasks:
  - cardiometabolic risk
  - hepatic risk
  - renal risk
  - integrated multi-system burden
- Compare baseline and stronger machine-learning models
- Generate feature importance and SHAP explanations
- Produce subgroup and scenario-style analyses
- Save figures, tables, and an outputs summary

## Expected Workflow
Step-0 output location:
- `/content/drive/MyDrive/Outputs/Smoking_NHANES_Step0/outputs/nhanes_smoking_project.csv`

Step-1 output location:
- `/content/drive/MyDrive/Outputs/Smoking_NHANES_Step1/`


## 1. Mount Google Drive and define output folders

This cell mounts Google Drive and prepares the Step-1 output directories.


In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Outputs/Smoking_NHANES_Step1')
else:
    BASE_DIR = Path('/content/Smoking_NHANES_Step1')

FIG_DIR = BASE_DIR / 'figures'
TAB_DIR = BASE_DIR / 'tables'
OTH_DIR = BASE_DIR / 'others'

for d in [BASE_DIR, FIG_DIR, TAB_DIR, OTH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('FIG_DIR:', FIG_DIR)
print('TAB_DIR:', TAB_DIR)
print('OTH_DIR:', OTH_DIR)


## 2. Import the required libraries

This cell imports the packages needed for preprocessing, modeling, evaluation, explainability, and artifact export.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    xgb_available = False

try:
    import shap
    shap_available = True
except Exception:
    shap_available = False

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

print("Libraries loaded.")
print("XGBoost available:", xgb_available)
print("SHAP available:", shap_available)


## 3. Load the Step-0 dataset automatically

This cell loads the integrated NHANES CSV from the Step-0 pipeline. It checks the most likely project locations.


In [ ]:
candidate_paths = [
    Path('/content/drive/MyDrive/Outputs/Smoking_NHANES_Step0/outputs/nhanes_smoking_project.csv'),
    Path('/content/drive/MyDrive/Smoking_NHANES_Step0/outputs/nhanes_smoking_project.csv'),
    Path('/content/nhanes_smoking_project.csv'),
    Path('/content/Smoking_NHANES_Step0/outputs/nhanes_smoking_project.csv'),
    Path('/mnt/data/nhanes_smoking_project.csv'),
]

DATA_PATH = None
for p in candidate_paths:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is not None:
    df = pd.read_csv(DATA_PATH)
    print('Dataset loaded successfully.')
    print('Loaded from:', DATA_PATH)
    print('Shape:', df.shape)
else:
    print('Dataset not found in the default locations.')
    print('Candidate paths checked:')
    for p in candidate_paths:
        print(' -', p)
    df = pd.DataFrame()

display(df.head())


## 4. Inspect shape, types, and missingness

This cell validates the imported dataset structure.


In [ ]:
if not df.empty:
    print('Shape:', df.shape)
    display(df.dtypes.to_frame('dtype').head(60))
    missing_df = df.isnull().mean().sort_values(ascending=False).mul(100).round(2).to_frame('missing_percent')
    display(missing_df.head(30))
else:
    print('Dataframe is empty.')


## 5. Rebuild derived organ targets if they are missing

This cell ensures that cardiometabolic, hepatic, renal, and integrated burden targets exist even if the CSV was saved without them.


In [ ]:
if not df.empty:
    print("Checking derived targets...")

    if 'cardiometabolic_risk_high' not in df.columns:
        print("Rebuilding cardiometabolic targets...")
        cm_conditions = []
        if 'bmi' in df.columns:
            cm_conditions.append(df['bmi'] >= 30)
        if 'systolic_bp' in df.columns:
            cm_conditions.append(df['systolic_bp'] >= 140)
        if 'diastolic_bp' in df.columns:
            cm_conditions.append(df['diastolic_bp'] >= 90)
        if 'glucose' in df.columns:
            cm_conditions.append(df['glucose'] >= 126)
        if 'triglycerides_mg_dl' in df.columns:
            cm_conditions.append(df['triglycerides_mg_dl'] >= 150)
        if len(cm_conditions) > 0:
            df['cardiometabolic_risk_proxy'] = np.sum(cm_conditions, axis=0)
            df['cardiometabolic_risk_high'] = (df['cardiometabolic_risk_proxy'] >= 2).astype(int)

    if 'hepatic_risk_high' not in df.columns:
        print("Rebuilding hepatic targets...")
        hepatic_conditions = []
        if 'alt' in df.columns:
            hepatic_conditions.append(df['alt'] > 40)
        if 'ast' in df.columns:
            hepatic_conditions.append(df['ast'] > 40)
        if 'ggt' in df.columns:
            hepatic_conditions.append(df['ggt'] > 60)
        if len(hepatic_conditions) > 0:
            df['hepatic_risk_proxy'] = np.sum(hepatic_conditions, axis=0)
            df['hepatic_risk_high'] = (df['hepatic_risk_proxy'] >= 1).astype(int)

    if 'renal_risk_high' not in df.columns:
        print("Rebuilding renal targets...")
        renal_conditions = []
        if 'egfr' in df.columns:
            renal_conditions.append(df['egfr'] < 60)
        if 'uacr_mg_g' in df.columns:
            renal_conditions.append(df['uacr_mg_g'] > 30)
        if 'creatinine' in df.columns:
            renal_conditions.append(df['creatinine'] > 1.3)
        if len(renal_conditions) > 0:
            df['renal_risk_proxy'] = np.sum(renal_conditions, axis=0)
            df['renal_risk_high'] = (df['renal_risk_proxy'] >= 1).astype(int)

    if 'multi_system_burden_high' not in df.columns:
        print("Rebuilding integrated burden target...")
        integrated_cols = [c for c in ['cardiometabolic_risk_high', 'hepatic_risk_high', 'renal_risk_high'] if c in df.columns]
        if len(integrated_cols) > 0:
            df['multi_system_burden_score'] = df[integrated_cols].sum(axis=1)
            df['multi_system_burden_high'] = (df['multi_system_burden_score'] >= 2).astype(int)

    print('Available derived targets:')
    for col in ['cardiometabolic_risk_high', 'hepatic_risk_high', 'renal_risk_high', 'multi_system_burden_high']:
        print(col, '->', col in df.columns)


## 6. Define the main feature groups

This cell groups the variables used in the notebook.


In [ ]:
smoking_target = 'smoking_exposure_group'

demographic_features = ['age', 'sex_label', 'race_ethnicity_label', 'education_label', 'poverty_income_ratio']
anthropometric_features = ['bmi', 'weight_kg', 'height_cm', 'waist_cm']
cardiometabolic_features = ['systolic_bp', 'diastolic_bp', 'glucose', 'triglycerides_mg_dl', 'total_cholesterol']
hepatic_features = ['alt', 'ast', 'ggt', 'alp']
renal_features = ['creatinine', 'bun', 'urine_albumin_mg_l', 'urine_creatinine_mg_dl', 'uacr_mg_g', 'egfr']

smoking_model_features = [c for c in (
    demographic_features +
    anthropometric_features +
    cardiometabolic_features +
    hepatic_features +
    renal_features
) if c in df.columns]

organ_targets = [c for c in ['cardiometabolic_risk_high', 'hepatic_risk_high', 'renal_risk_high', 'multi_system_burden_high'] if c in df.columns]

print('Smoking model features:')
print(smoking_model_features)
print('Organ targets:')
print(organ_targets)


## 7. Inspect class balance

This cell checks the smoking and organ-system target distributions.


In [ ]:
for col in [smoking_target] + organ_targets:
    if col in df.columns:
        print('=' * 80)
        print(col)
        display(df[col].value_counts(dropna=False).to_frame('count'))


## 8. Prepare the smoking-exposure modeling cohort

This cell selects features and target for smoking classification.


In [ ]:
print("Preparing smoking modeling cohort...")

if not df.empty and smoking_target in df.columns:
    smoking_df = df[smoking_model_features + [smoking_target]].copy()
    smoking_df = smoking_df.dropna(subset=[smoking_target])

    print("✅ smoking_df created")
    print("Shape:", smoking_df.shape)
    display(smoking_df.head())
else:
    smoking_df = pd.DataFrame()
    print("❌ Cannot create smoking_df")
    print("Check whether df is loaded and smoking_exposure_group exists.")


## 9. Build preprocessing pipelines

This cell prepares numerical and categorical transforms.


In [ ]:
print("Building preprocessing objects...")

if not smoking_df.empty:
    X = smoking_df.drop(columns=[smoking_target])
    y = smoking_df[smoking_target]

    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    preprocessor = ColumnTransformer(transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    print("✅ Train/test split done")
    print("X_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)
    print("Classes:", list(label_encoder.classes_))
else:
    print("Smoking cohort is empty. Skipping preprocessing.")


## 10. Train a logistic-regression baseline

This cell fits a transparent baseline model.


In [ ]:
print("Training Logistic Regression...")

if not smoking_df.empty:
    lr_model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1500))
    ])
    lr_model.fit(X_train, y_train)
    y_pred_lr = lr_model.predict(X_test)

    print('✅ Logistic Regression trained')
    print('Accuracy:', round(accuracy_score(y_test, y_pred_lr), 4))
    print('Weighted F1:', round(f1_score(y_test, y_pred_lr, average='weighted'), 4))
    print(classification_report(y_test, y_pred_lr, target_names=label_encoder.classes_))
else:
    print("Smoking cohort is empty. Logistic Regression skipped.")


## 11. Train a random-forest model

This cell fits a nonlinear tree-based model.


In [ ]:
print("Training Random Forest...")

if not smoking_df.empty:
    rf_model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=400,
            random_state=42,
            class_weight='balanced_subsample',
            n_jobs=-1
        ))
    ])
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)

    print('✅ Random Forest trained')
    print('Accuracy:', round(accuracy_score(y_test, y_pred_rf), 4))
    print('Weighted F1:', round(f1_score(y_test, y_pred_rf, average='weighted'), 4))
    print(classification_report(y_test, y_pred_rf, target_names=label_encoder.classes_))

    fitted_preprocessor = rf_model.named_steps['preprocessor']
    fitted_rf = rf_model.named_steps['classifier']
else:
    print("Smoking cohort is empty. Random Forest skipped.")


## 12. Train an XGBoost model if available

This cell fits a stronger boosting model.


In [ ]:
print("Training XGBoost if available...")

if not smoking_df.empty and xgb_available:
    xgb_model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='mlogloss',
            random_state=42
        ))
    ])
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)

    print('✅ XGBoost trained')
    print('Accuracy:', round(accuracy_score(y_test, y_pred_xgb), 4))
    print('Weighted F1:', round(f1_score(y_test, y_pred_xgb, average='weighted'), 4))
    print(classification_report(y_test, y_pred_xgb, target_names=label_encoder.classes_))
else:
    print('XGBoost not available or smoking dataset not ready.')


## 13. Summarize smoking-classification performance

This cell compares the main classification models.


In [ ]:
print("Summarizing classification models...")

results = []
if not smoking_df.empty:
    results.append({'model': 'LogisticRegression', 'accuracy': accuracy_score(y_test, y_pred_lr), 'weighted_f1': f1_score(y_test, y_pred_lr, average='weighted')})
    results.append({'model': 'RandomForest', 'accuracy': accuracy_score(y_test, y_pred_rf), 'weighted_f1': f1_score(y_test, y_pred_rf, average='weighted')})
    if xgb_available:
        results.append({'model': 'XGBoost', 'accuracy': accuracy_score(y_test, y_pred_xgb), 'weighted_f1': f1_score(y_test, y_pred_xgb, average='weighted')})

    results_df = pd.DataFrame(results).sort_values('weighted_f1', ascending=False)
    display(results_df)
    results_df.to_csv(TAB_DIR / 'smoking_classification_results.csv', index=False)
else:
    print("Smoking cohort is empty. No results to summarize.")


## 14. Plot the confusion matrix for the best model

This cell visualizes where smoking classes are confused.


In [ ]:
print("Generating confusion matrix...")

if not smoking_df.empty and 'results_df' in globals() and not results_df.empty:
    best_model_name = results_df.iloc[0]['model']
    if best_model_name == 'XGBoost' and xgb_available:
        best_pred = y_pred_xgb
    elif best_model_name == 'RandomForest':
        best_pred = y_pred_rf
    else:
        best_pred = y_pred_lr

    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_test, best_pred,
        display_labels=label_encoder.classes_,
        cmap='Blues',
        xticks_rotation=45,
        ax=ax
    )
    plt.title(f'Confusion Matrix - {best_model_name}')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'confusion_matrix_smoking.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Confusion matrix saved")
else:
    print("Confusion matrix skipped.")


## 15. Evaluate a binary current-smoker calibration example

This cell assesses a binary current-smoker vs non-current-smoker formulation.


In [ ]:
print("Running calibration analysis...")

if not smoking_df.empty:
    binary_df = smoking_df.copy()
    binary_df['current_smoker_binary'] = binary_df[smoking_target].isin(['current_light', 'current_heavy', 'current_unspecified']).astype(int)

    X_bin = binary_df.drop(columns=[smoking_target, 'current_smoker_binary'])
    y_bin = binary_df['current_smoker_binary']

    cat_bin = X_bin.select_dtypes(include=['object', 'category']).columns.tolist()
    num_bin = [c for c in X_bin.columns if c not in cat_bin]

    pre_bin = ColumnTransformer(transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_bin),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_bin)
    ])

    Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin)
    calib_model = Pipeline(steps=[('preprocessor', pre_bin), ('classifier', LogisticRegression(max_iter=1500))])
    calib_model.fit(Xb_train, yb_train)
    yb_prob = calib_model.predict_proba(Xb_test)[:, 1]
    yb_pred = (yb_prob >= 0.5).astype(int)

    print('Binary current-smoker Accuracy:', round(accuracy_score(yb_test, yb_pred), 4))
    print('Binary current-smoker F1:', round(f1_score(yb_test, yb_pred), 4))
    print('Binary current-smoker Brier Score:', round(brier_score_loss(yb_test, yb_prob), 4))

    frac_pos, mean_pred = calibration_curve(yb_test, yb_prob, n_bins=10)
    plt.figure(figsize=(6, 5))
    plt.plot(mean_pred, frac_pos, marker='o', label='Model')
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
    plt.xlabel('Mean predicted probability')
    plt.ylabel('Fraction of positives')
    plt.title('Calibration Curve: Current Smoker vs Non-Current')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'calibration_current_smoker.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Calibration figure saved")
else:
    print("Calibration analysis skipped.")


## 16. Estimate feature importance

This cell extracts random-forest feature importance for smoking classification.


In [ ]:
print("Computing feature importance...")

if not smoking_df.empty and 'fitted_rf' in globals() and 'fitted_preprocessor' in globals():
    feature_names = []
    if numeric_cols:
        feature_names.extend(numeric_cols)
    if categorical_cols:
        ohe = fitted_preprocessor.named_transformers_['cat'].named_steps['onehot']
        feature_names.extend(ohe.get_feature_names_out(categorical_cols).tolist())

    importance_df = pd.DataFrame({'feature': feature_names, 'importance': fitted_rf.feature_importances_}).sort_values('importance', ascending=False)
    display(importance_df.head(20))
    importance_df.to_csv(TAB_DIR / 'rf_feature_importance_smoking.csv', index=False)

    topk = importance_df.head(15).iloc[::-1]
    plt.figure(figsize=(8, 6))
    plt.barh(topk['feature'], topk['importance'])
    plt.title('Top Random-Forest Feature Importances')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'rf_feature_importance_smoking.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Feature importance saved")
else:
    print("Feature importance skipped.")


## 17. Generate SHAP explanations if available

This cell computes SHAP explanations for the random-forest smoking model with progress prints.


In [ ]:
print("=== SHAP EXPLANATION START ===")

if 'smoking_df' not in globals():
    print("❌ smoking_df not found. Run the smoking dataset preparation cell first.")

elif 'fitted_rf' not in globals():
    print("❌ fitted_rf not found. Run the Random Forest training cell first.")

elif 'fitted_preprocessor' not in globals():
    print("❌ fitted_preprocessor not found. Run the preprocessing/training cell first.")

elif 'X_test' not in globals():
    print("❌ X_test not found. Run the train/test split cell first.")

elif not shap_available:
    print("❌ SHAP not available.")

elif smoking_df.empty:
    print("❌ smoking_df is empty.")

else:
    print("✅ All prerequisites available. Starting SHAP...")

    print("Step 1/6: Transforming X_test...")
    X_test_proc = fitted_preprocessor.transform(X_test)

    print("Step 2/6: Converting to dense if needed...")
    if hasattr(X_test_proc, 'toarray'):
        X_test_shap = X_test_proc.toarray()
    else:
        X_test_shap = X_test_proc

    print(f"Shape: {X_test_shap.shape}")

    print("Step 3/6: Sampling...")
    n_samples = min(200, X_test_shap.shape[0])
    X_test_sample = X_test_shap[:n_samples]
    print(f"Using {n_samples} samples")

    print("Step 4/6: Building explainer...")
    explainer = shap.TreeExplainer(fitted_rf)

    print("Step 5/6: Computing SHAP values...")
    shap_values = explainer.shap_values(X_test_sample)
    print("SHAP computed")

    print("Step 6/6: Plotting...")
    try:
        if isinstance(shap_values, list):
            shap.summary_plot(shap_values[0], X_test_sample, feature_names=feature_names, show=False)
        else:
            shap.summary_plot(shap_values, X_test_sample, feature_names=feature_names, show=False)

        plt.tight_layout()
        plt.savefig(FIG_DIR / 'shap_summary_smoking.png', dpi=300, bbox_inches='tight')
        plt.show()

        print("=== SHAP DONE ===")

    except Exception as e:
        print("SHAP plot error:", e)


## 18. Evaluate subgroup performance

This cell checks performance variation across demographic strata.


In [ ]:
print("Running subgroup analysis...")

if not smoking_df.empty and 'best_pred' in globals():
    subgroup_df = X_test.copy()
    subgroup_df['true'] = y_test
    subgroup_df['pred'] = best_pred

    subgroup_rows = []
    for subgroup_col in ['sex_label', 'race_ethnicity_label']:
        if subgroup_col in subgroup_df.columns:
            for subgroup_value, sub_df in subgroup_df.groupby(subgroup_col):
                print(f"Processing subgroup {subgroup_col} = {subgroup_value} (n={len(sub_df)})")
                if len(sub_df) < 20:
                    continue
                subgroup_rows.append({
                    'subgroup_variable': subgroup_col,
                    'subgroup_value': subgroup_value,
                    'n': len(sub_df),
                    'accuracy': accuracy_score(sub_df['true'], sub_df['pred']),
                    'weighted_f1': f1_score(sub_df['true'], sub_df['pred'], average='weighted')
                })

    subgroup_results_df = pd.DataFrame(subgroup_rows)
    display(subgroup_results_df)
    if not subgroup_results_df.empty:
        subgroup_results_df.to_csv(TAB_DIR / 'subgroup_results_smoking.csv', index=False)
        print("✅ Subgroup results saved")
else:
    print("Subgroup analysis skipped.")


## 19. Train organ-specific risk models

This cell creates parallel prediction tasks for cardiometabolic, hepatic, renal, and integrated burden outcomes using robust checks and progress prints.


In [ ]:
print("Starting organ-specific modeling...")

organ_results = []
organ_models = {}

for i, target in enumerate(organ_targets, start=1):
    print(f"\n[{i}/{len(organ_targets)}] Target: {target}")

    feature_cols = [c for c in smoking_model_features if c in df.columns]
    task_df = df[feature_cols + [target]].copy().dropna(subset=[target])

    if task_df.empty:
        print(f"Skipping {target}: empty dataset")
        continue

    Xt = task_df.drop(columns=[target])
    yt = task_df[target].astype(int)

    if yt.nunique() < 2:
        print(f"Skipping {target}: only one class present")
        continue

    cat_cols = Xt.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = [c for c in Xt.columns if c not in cat_cols]

    print(" - Building preprocessing pipeline")
    pre_task = ColumnTransformer(transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
    ])

    print(" - Splitting train/test")
    Xt_train, Xt_test, yt_train, yt_test = train_test_split(
        Xt, yt, test_size=0.2, random_state=42, stratify=yt
    )

    print(" - Training Random Forest")
    task_model = Pipeline(steps=[
        ('preprocessor', pre_task),
        ('classifier', RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight='balanced_subsample',
            n_jobs=-1
        ))
    ])

    task_model.fit(Xt_train, yt_train)

    print(" - Predicting")
    yt_pred = task_model.predict(Xt_test)
    yt_prob = task_model.predict_proba(Xt_test)[:, 1]

    try:
        roc_auc = roc_auc_score(yt_test, yt_prob)
    except Exception:
        roc_auc = np.nan

    organ_results.append({
        'target': target,
        'n': len(task_df),
        'accuracy': accuracy_score(yt_test, yt_pred),
        'f1': f1_score(yt_test, yt_pred),
        'roc_auc': roc_auc
    })

    organ_models[target] = {
        'model': task_model,
        'X_test': Xt_test,
        'y_test': yt_test,
        'y_pred': yt_pred,
        'y_prob': yt_prob
    }

    print(f" - Done: accuracy={accuracy_score(yt_test, yt_pred):.4f}, f1={f1_score(yt_test, yt_pred):.4f}, roc_auc={roc_auc:.4f}" if pd.notna(roc_auc) else f" - Done: accuracy={accuracy_score(yt_test, yt_pred):.4f}, f1={f1_score(yt_test, yt_pred):.4f}, roc_auc=nan")

if len(organ_results) > 0:
    organ_results_df = pd.DataFrame(organ_results)
    if 'roc_auc' in organ_results_df.columns:
        organ_results_df = organ_results_df.sort_values('roc_auc', ascending=False)
    display(organ_results_df)
    organ_results_df.to_csv(TAB_DIR / 'organ_risk_results.csv', index=False)
    print("✅ Organ risk results saved")
else:
    print('No valid organ models were trained.')
    organ_results_df = pd.DataFrame()


## 20. Compare smoking and organ-system tasks

This cell summarizes the predictive landscape across tasks.


In [ ]:
print("Combining task summaries...")

if not smoking_df.empty and 'results_df' in globals() and not results_df.empty:
    smoking_best = results_df.iloc[0].copy()
    smoking_best_df = pd.DataFrame([{
        'target': 'smoking_exposure_group',
        'n': len(smoking_df),
        'accuracy': smoking_best['accuracy'],
        'f1': smoking_best['weighted_f1'],
        'roc_auc': np.nan
    }])

    combined_summary_df = pd.concat([smoking_best_df, organ_results_df], ignore_index=True)
    display(combined_summary_df)
    combined_summary_df.to_csv(TAB_DIR / 'combined_task_summary.csv', index=False)
    print("✅ Combined task summary saved")
else:
    print("Combined summary skipped.")


## 21. Create intervention-style scenario analysis

This cell performs a simple what-if analysis for integrated burden.


In [ ]:
print("Running scenario analysis...")

if 'multi_system_burden_high' in organ_models:
    bundle = organ_models['multi_system_burden_high']
    model = bundle['model']
    sample_case = bundle['X_test'].iloc[[0]].copy()
    modified_case = sample_case.copy()

    if 'bmi' in modified_case.columns and pd.notna(modified_case['bmi'].iloc[0]):
        modified_case.loc[:, 'bmi'] = max(18.5, modified_case['bmi'].iloc[0] - 3)
    if 'systolic_bp' in modified_case.columns and pd.notna(modified_case['systolic_bp'].iloc[0]):
        modified_case.loc[:, 'systolic_bp'] = max(100, modified_case['systolic_bp'].iloc[0] - 10)
    if 'triglycerides_mg_dl' in modified_case.columns and pd.notna(modified_case['triglycerides_mg_dl'].iloc[0]):
        modified_case.loc[:, 'triglycerides_mg_dl'] = max(50, modified_case['triglycerides_mg_dl'].iloc[0] - 30)

    original_prob = model.predict_proba(sample_case)[0, 1]
    modified_prob = model.predict_proba(modified_case)[0, 1]

    scenario_df = pd.DataFrame({
        'scenario': ['original', 'modified'],
        'predicted_multi_system_burden_probability': [original_prob, modified_prob]
    })
    display(scenario_df)
    scenario_df.to_csv(TAB_DIR / 'scenario_analysis_multi_system_burden.csv', index=False)

    plt.figure(figsize=(5, 4))
    plt.bar(scenario_df['scenario'], scenario_df['predicted_multi_system_burden_probability'])
    plt.title('Scenario Analysis: Multi-System Burden Probability')
    plt.ylabel('Predicted probability')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'scenario_analysis_multi_system_burden.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Scenario analysis saved")
else:
    print('Integrated burden model not available.')


## 22. Save an outputs summary

This cell writes a short text summary for reproducibility and article writing.


In [ ]:
summary_lines = [
    'Step-1 NHANES Modeling Notebook',
    '',
    f'Loaded dataset shape: {df.shape if not df.empty else "N/A"}',
    f'Smoking modeling cohort shape: {smoking_df.shape if "smoking_df" in globals() and not smoking_df.empty else "N/A"}',
    '',
    'Generated outputs:',
    '- smoking classification results',
    '- organ-risk prediction results',
    '- combined task summary',
    '- feature importance table',
    '- subgroup analysis if available',
    '- confusion matrix figure',
    '- calibration figure',
    '- SHAP figure if available',
    '- scenario analysis figure'
]

summary_path = OTH_DIR / 'outputs_summary_step1.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))

print('Saved summary to:', summary_path)


## 23. List generated artifacts

This cell lists the main saved files.


In [ ]:
for folder in [FIG_DIR, TAB_DIR, OTH_DIR]:
    print('=' * 80)
    print(folder)
    if folder.exists():
        for item in sorted(folder.iterdir()):
            print(item.name)
